Pulls real VisDrone-MOT video frames (real scene_id, frame_number, and
detections -- including each object's real cross-frame tracking index)
via the fiftyone library and lands them in raw.

Earlier attempts at this exact dataset (Voxel51/visdrone-mot) failed when
read through Hugging Face's auto-converted Parquet view -- that conversion
silently drops FiftyOne's nested scene_id/frame_number/detections fields
down to a bare `image` column. Loading through the real fiftyone library
instead preserves everything. Getting fiftyone running required adding a
real MongoDB service to docker-compose.yml -- see README.

Note: an earlier version of this script called load_from_hub with no
limit, which tries to download the FULL dataset (~2,847 images across
all 7 real scenes) before filtering down to just the scenes we want --
that triggered a 429 Too Many Requests from HF's Xet storage backend.
MAX_SAMPLES bounds the download, and since we don't know in advance
which scenes will land inside a capped download, we pick whichever
real scenes actually show up rather than hardcoding scene names.

Standard library imports for JSON encoding, byte buffers, and retry backoff timing.

In [1]:
import json
import io
import time

Imports for S3/RustFS access, DuckDB, loading the real dataset via fiftyone, and image handling.

In [2]:
import boto3
import duckdb
import fiftyone.utils.huggingface as fouh
import pandas as pd
from fiftyone import ViewField as F
from PIL import Image

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration: a bounded sample cap (to avoid rate limits), how many real scenes to target, and how many frames per scene.

In [3]:
BUCKET = "lakehouse"
S3_PREFIX = "assets/visdrone/images"
REPO_ID = "Voxel51/visdrone-mot"

Configuration: a bounded sample cap (to avoid rate limits), how many real scenes to target, and how many frames per scene.

In [4]:
# bounded download -- big enough to comfortably span at least a couple of
# real scenes, small enough to avoid hammering HF's rate limits
MAX_SAMPLES = 500
N_SCENES = 2
FRAMES_PER_SCENE = 30

Creates a boto3 S3 client pointed at RustFS, same as the COCO script.

In [5]:
# connects to RustFS the same way the COCO script does
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url="http://rustfs:9000",
        aws_access_key_id="rustfsadmin",
        aws_secret_access_key="rustfsadmin",
    )

Connects DuckDB and attaches the DuckLake catalog.

In [6]:
# turns on DuckLake, connects it to RustFS, opens the catalog
def attach_lakehouse():
    con = duckdb.connect()
    con.execute(open("sql/00_attach.sql").read())
    return con

Loads a bounded slice of the VisDrone-MOT dataset via fiftyone, retrying with backoff if HF rate-limits the download.

In [7]:
# loads a bounded slice of the FiftyOne dataset (not the auto-converted
# Parquet mirror, which drops the nested fields we actually need), with
# a few retries since HF's Xet backend can rate-limit transiently
def load_visdrone_dataset(max_retries=3):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return fouh.load_from_hub(REPO_ID, max_samples=MAX_SAMPLES)
        except Exception as e:
            last_error = e
            wait = 10 * attempt
            print(f"Download attempt {attempt} failed ({e}); retrying in {wait}s...")
            time.sleep(wait)
    raise last_error

Picks whichever real scenes actually landed in the bounded download, rather than hardcoding scene names that might be missing.

In [8]:
# picks the first N distinct real scenes actually present in the
# downloaded subset, in the order they first appear -- avoids hardcoding
# scene names that might not have made it into a rate-limit-bounded download
def select_target_scenes(dataset, n_scenes):
    seen = []
    for sample in dataset.select_fields("scene_id"):
        if sample.scene_id not in seen:
            seen.append(sample.scene_id)
        if len(seen) >= n_scenes:
            break
    return seen

Selects exactly FRAMES_PER_SCENE samples from one real scene, sorted by real frame_number, so the result is a genuine contiguous run.

In [9]:
# pulls exactly FRAMES_PER_SCENE samples from one real scene, ordered by
# real frame_number, so the frames we land are a genuine contiguous run
def select_scene_frames(dataset, scene_id):
    view = dataset.match(F("scene_id") == scene_id).sort_by("frame_number")
    return list(view.limit(FRAMES_PER_SCENE))

Uploads one frame's image to RustFS and builds its metadata row, keeping each detection's real cross-frame tracking ID. Handles both possible shapes of the detections field -- a wrapper object or a bare list -- after an earlier version silently discarded real detections by treating the bare-list case as empty.

In [10]:
# uploads one frame's image to RustFS and builds its metadata row, keeping
# the real detections -- including each object's real cross-frame tracking
# `index` -- as JSON
def upload_frame_and_build_row(s3, index, sample):
    img = Image.open(sample.filepath).convert("RGB")

    key = f"{S3_PREFIX}/{index:04d}.jpg"
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf, ContentType="image/jpeg")

    # sample.detections' actual shape depends on how the field was stored --
    # sometimes it's a proper Detections wrapper (has .detections), but for
    # this dataset it comes back as a bare list of Detection objects
    # directly. An earlier version of this code treated that bare-list case
    # as "no detections" and silently discarded every real detection --
    # confirmed by raw.visdrone_frames landing with 0 total detections.
    # Fixed: fall back to treating it as the flat list it actually is.
    raw_detections = sample.detections
    if raw_detections is None:
        detection_objects = []
    elif hasattr(raw_detections, "detections"):
        detection_objects = raw_detections.detections
    else:
        detection_objects = raw_detections

    detections = [
        {
            "label": det.label,
            "bounding_box": det.bounding_box,
            "track_id": det.index,
            "occlusion": det.occlusion,
            "visibility": det.visibility,
        }
        for det in detection_objects
    ]

    return {
        "image_uri": f"s3://{BUCKET}/{key}",
        "width": img.width,
        "height": img.height,
        "scene_id": sample.scene_id,
        "frame_number": sample.frame_number,
        "n_detections": len(detections),
        "detections_json": json.dumps(detections),
    }

Connects to RustFS and attaches the DuckLake catalog.

In [11]:
s3 = make_s3_client()

con = attach_lakehouse()

Loads a bounded slice of the VisDrone-MOT dataset via fiftyone and confirms how many samples came through.

In [12]:
print(f"Loading up to {MAX_SAMPLES} samples from {REPO_ID} via fiftyone...")

dataset = load_visdrone_dataset()

print(f"Dataset loaded: {len(dataset)} samples")

Loading up to 500 samples from Voxel51/visdrone-mot via fiftyone...


Loading dataset


Importing samples...


   0% ||----------------|   1/500 [8.4ms elapsed, 4.2s remaining, 118.7 samples/s] 

 100% |█████████████████| 500/500 [121.8ms elapsed, 0s remaining, 4.1K samples/s]  

...

 60%|██████    | 3/5 [00:14<00:09,  4.94s/it]

 80%|████████  | 4/5 [00:19<00:04,  4.99s/it]

100%|██████████| 5/5 [00:23<00:00,  4.80s/it]

100%|██████████| 5/5 [00:23<00:00,  4.79s/it]

Dataset loaded: 500 samples


Picks the real scenes actually present in this download.

In [13]:
scene_ids = select_target_scenes(dataset, N_SCENES)

print(f"Real scenes found in this download: {scene_ids}")

Real scenes found in this download: ['uav0000086_00000_v', 'uav0000182_00000_v']


Loops over the target scenes, selecting and uploading a contiguous run of real frames from each.

In [14]:
rows = []

index = 0

for scene_id in scene_ids:
    print(f"\nSelecting {FRAMES_PER_SCENE} frames from real scene {scene_id}...")
    samples = select_scene_frames(dataset, scene_id)
    print(f"Found {len(samples)} frames")
    for sample in samples:
        rows.append(upload_frame_and_build_row(s3, index, sample))
        index += 1
    print(f"  ...{index} frames uploaded so far")


Selecting 30 frames from real scene uav0000086_00000_v...


Found 30 frames


  ...30 frames uploaded so far

Selecting 30 frames from real scene uav0000182_00000_v...


Found 30 frames


  ...60 frames uploaded so far


Confirms all real frames were uploaded to RustFS.

In [15]:
print(f"\nUploaded {len(rows)} real frames to s3://{BUCKET}/{S3_PREFIX}/")


Uploaded 60 real frames to s3://lakehouse/assets/visdrone/images/


Writes the collected metadata into `raw.visdrone_frames`, creating a new DuckLake snapshot.

In [16]:
df = pd.DataFrame(rows)

con.register("visdrone_df", df)

con.execute("CREATE OR REPLACE TABLE raw.visdrone_frames AS SELECT * FROM visdrone_df")

Confirms the row count and total real detections landed in the table.

In [17]:
count = con.sql("SELECT COUNT(*) FROM raw.visdrone_frames").fetchone()[0]

total_detections = con.sql("SELECT SUM(n_detections) FROM raw.visdrone_frames").fetchone()[0]

print(f"raw.visdrone_frames now has {count} rows, {total_detections} total real detections")

raw.visdrone_frames now has 60 rows, 1995 total real detections


Shows how many frames and detections landed in each real scene.

In [18]:
print("\nFrames per (real) scene:")

con.sql("""
    SELECT scene_id, COUNT(*) AS n_frames, SUM(n_detections) AS n_detections
    FROM raw.visdrone_frames GROUP BY scene_id ORDER BY scene_id
""").show()


Frames per (real) scene:
┌────────────────────┬──────────┬──────────────┐
│      scene_id      │ n_frames │ n_detections │
│      varchar       │  int64   │    int128    │
├────────────────────┼──────────┼──────────────┤
│ uav0000086_00000_v │       30 │         1080 │
│ uav0000182_00000_v │       30 │          915 │
└────────────────────┴──────────┴──────────────┘



Shows the most recent DuckLake snapshots as proof a new version was created.

In [19]:
print("\nMost recent snapshots:")

con.sql("FROM ducklake_snapshots('lake') ORDER BY snapshot_id DESC LIMIT 5").show()


Most recent snapshots:
┌─────────────┬───────────────────────────────┬────────────────┬───────────────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────────┐
│ snapshot_id │         snapshot_time         │ schema_version │                              changes                              │ author  │ commit_message │ commit_extra_info │
│    int64    │   timestamp with time zone    │     int64      │                      map(varchar, varchar[])                      │ varchar │    varchar     │      varchar      │
├─────────────┼───────────────────────────────┼────────────────┼───────────────────────────────────────────────────────────────────┼─────────┼────────────────┼───────────────────┤
│           6 │ 2026-08-09 05:13:18.071938+00 │              5 │ {tables_created=[raw.visdrone_frames], tables_inserted_into=[5]}  │ NULL    │ NULL           │ NULL              │
│           5 │ 2026-08-09 05:12:38.965846+00 │              4 │ {tables_ins